# Privacy-Preserving Computing: k-Anonymity, l-Diversity and t-Closeness

**TU Berlin | PPC Chapter 2 | Interactive demonstration**

This notebook uses a **synthetic** dataset. It separates *data transformation* (pandas) from *privacy-model evaluation* (pyCANON). The metrics alone do not anonymize data.

Run the cells from top to bottom. No external dataset is needed.

## What is pyCANON?

**pyCANON** is an open-source Python library for evaluating tabular datasets against privacy models. You provide a **pandas DataFrame** and specify which columns are quasi-identifiers (here: age and ZIP code) and which are sensitive attributes (here: diagnosis).

The library calculates metrics including:

- **k-Anonymity:** What is the size of the smallest group sharing the same quasi-identifiers?
- **Distinct l-Diversity:** What is the minimum number of distinct sensitive values in any group?
- **t-Closeness:** How much does a group's sensitive-value distribution differ from the distribution in the full dataset?

You can use these metrics to identify weaknesses and compare different generalization or suppression strategies. In this notebook, **pandas** transforms the data and **pyCANON** measures the effects. The library also provides reports and additional privacy metrics.

**Important:** pyCANON does not anonymize the data itself. Results depend on your choice of quasi-identifiers and sensitive attributes, and do not by themselves establish that a dataset is legally anonymous.

Developed by Judith Sáinz-Pardo Díaz and Álvaro López García (IFCA–CSIC).

[GitHub: IFCA-Advanced-Computing/pycanon](https://github.com/IFCA-Advanced-Computing/pycanon) · [Documentation](https://pycanon.readthedocs.io/en/stable/)


## Python version: use Python 3.12 for this demo

**Use a Python 3.12 notebook kernel with the package versions used in this demo.** With pyCANON 1.3.6, installation on Python 3.14 can spend a long time at `Preparing metadata (pyproject.toml)` and then fail.

The reason is that pyCANON 1.3.6 pins older dependencies: `numpy==2.0.2` supports Python 3.9–3.12, and `scipy==1.15.3` supports Python 3.10–3.13. Python 3.12 is supported by both. On Python 3.14, pip may download source archives (`.tar.gz`) and attempt unsupported source builds instead of installing prebuilt wheels. Waiting longer or restarting the same kernel does not resolve this compatibility issue.

Check the Python version of your current kernel:

```python
import sys
print(sys.version)
```

If you already have Python 3.12 installed, create an environment from your project terminal:

```bash
python3.12 -m venv .venv312
.venv312/bin/python -m pip install pycanon==1.3.6 pandas matplotlib ipykernel
```

Alternatively, if you use [uv](https://docs.astral.sh/uv/), it can install Python 3.12 and create the environment:

```bash
uv python install 3.12
uv venv --python 3.12 --seed .venv312
uv pip install --python .venv312/bin/python pycanon==1.3.6 pandas matplotlib ipykernel
```

In **VS Code**, choose **Select Kernel → Python Environments → `.venv312`** in the notebook toolbar. Installing a separate Python version does not automatically switch the notebook kernel. You can keep your system Python unchanged.

This note applies to **pyCANON 1.3.6**; newer releases may change these dependency requirements. See the [NumPy 2.0.2 release notes](https://numpy.org/devdocs/release/2.0.2-notes.html), [SciPy compatibility table](https://docs.scipy.org/doc/scipy/dev/toolchain.html), and [pyCANON 1.3.6](https://pypi.org/project/pycanon/1.3.6/).


In [16]:
%pip install -q pycanon pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd
from pycanon import anonymity
from IPython.display import display

raw = pd.DataFrame({
    "Name": ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank", "Grace", "Heidi"],
    "Age": [28, 29, 27, 28, 46, 47, 45, 46],
    "ZIP": ["10115", "10117", "10119", "10115", "10551", "10553", "10555", "10551"],
    "Condition": ["Diabetes", "Diabetes", "Diabetes", "Diabetes", "Flu", "Migraine", "Flu", "Migraine"]
})
display(raw)

,Name,Age,ZIP,Condition
0,Alice,28,10115,Diabetes
1,Bob,29,10117,Diabetes
2,Carol,27,10119,Diabetes
3,Dave,28,10115,Diabetes
4,Eve,46,10551,Flu
5,Frank,47,10553,Migraine
6,Grace,45,10555,Flu
7,Heidi,46,10551,Migraine


## 1. Quasi-identifiers and equivalence classes

Removing `Name` does not make records indistinguishable. `Age` and `ZIP` may still distinguish individuals. In this example, **Condition** is the sensitive attribute.

In [18]:
QI = ["Age", "ZIP"]
SA = ["Condition"]

def report(df, qi=QI, sa=SA):
    print("k-anonymity:", anonymity.k_anonymity(df, qi))
    print("distinct l-diversity:", anonymity.l_diversity(df, qi, sa))
    print("t-closeness:", round(anonymity.t_closeness(df, qi, sa), 4))
    display(df.groupby(qi, dropna=False).agg(
        records=(sa[0], "size"),
        sensitive_values=(sa[0], lambda x: ", ".join(sorted(set(x))))
    ).reset_index())

identified_removed = raw.drop(columns="Name")
report(identified_removed)

k-anonymity: 1
distinct l-diversity: 1
t-closeness: 0.75


,Age,ZIP,records,sensitive_values
0,27,10119,1,Diabetes
1,28,10115,2,Diabetes
2,29,10117,1,Diabetes
3,45,10555,1,Flu
4,46,10551,2,"Flu, Migraine"
5,47,10553,1,Migraine


## 2. Generalization improves k-anonymity

We generalize ages into two ranges and ZIP codes into one region. The resulting equivalence classes contain four records each.

In [19]:
generalized = identified_removed.copy()
generalized["Age"] = generalized["Age"].map(lambda x: "20-39" if x < 40 else "40-59")
generalized["ZIP"] = "Berlin"
display(generalized)
report(generalized)

,Age,ZIP,Condition
0,20-39,Berlin,Diabetes
1,20-39,Berlin,Diabetes
2,20-39,Berlin,Diabetes
3,20-39,Berlin,Diabetes
4,40-59,Berlin,Flu
5,40-59,Berlin,Migraine
6,40-59,Berlin,Flu
7,40-59,Berlin,Migraine


k-anonymity: 4
distinct l-diversity: 1
t-closeness: 0.5


,Age,ZIP,records,sensitive_values
0,20-39,Berlin,4,Diabetes
1,40-59,Berlin,4,"Flu, Migraine"


## 3. Homogeneity attack

The first age group is entirely associated with **Diabetes**. A group can be k-anonymous but disclose its sensitive value. Distinct l-diversity catches this problem.

In [20]:
display(pd.crosstab(generalized["Age"], generalized["Condition"], normalize="index").round(2))

Condition,Diabetes,Flu,Migraine
Age,,,
20-39,1.0,0.0,0.0
40-59,0.0,0.5,0.5


## 4. Increase distinct l-diversity

For demonstration, we create a **different synthetic dataset** with two sensitive values in both groups. We do **not** recommend changing real diagnoses to meet a metric. In real anonymization, change QIs through generalization/suppression or use an appropriate release design.

In [21]:
diverse = generalized.copy()
diverse["Condition"] = ["Diabetes", "Diabetes", "Flu", "Flu", "Flu", "Migraine", "Flu", "Migraine"]
report(diverse)

k-anonymity: 4
distinct l-diversity: 2
t-closeness: 0.25


,Age,ZIP,records,sensitive_values
0,20-39,Berlin,4,"Diabetes, Flu"
1,40-59,Berlin,4,"Flu, Migraine"


## 5. t-Closeness compares local and global distributions

Even when each group contains two sensitive values, their proportions can differ strongly from the dataset-wide distribution. Compare both groups with the global baseline.

In [22]:
global_dist = diverse["Condition"].value_counts(normalize=True).sort_index()
local_dist = pd.crosstab(diverse["Age"], diverse["Condition"], normalize="index").reindex(columns=global_dist.index, fill_value=0)
print("Global distribution:")
display(global_dist.rename("share").to_frame())
print("Per-group distributions:")
display(local_dist.round(2))
print("pyCANON t-closeness:", anonymity.t_closeness(diverse, QI, SA))

Global distribution:


,share
Condition,
Diabetes,0.25
Flu,0.50
Migraine,0.25


Per-group distributions:


Condition,Diabetes,Flu,Migraine
Age,,,
20-39,0.5,0.5,0.0
40-59,0.0,0.5,0.5


pyCANON t-closeness: 0.25


## 6. Experiment

1. Change the quasi-identifiers to `QI = ["Age"]`. How does k change?
2. Change one sensitive value in the first equivalence class. What happens to l?
3. Create groups with the same sensitive-value distribution as the full dataset. What happens to t?
4. Explain why larger k does **not** imply larger l or smaller t.

**Important:** k-anonymity, l-diversity, and t-closeness depend on the chosen quasi-identifiers, sensitive attributes, and threat model. They do not by themselves establish that a released dataset is legally anonymous.

## Further reading

- [pyCANON documentation](https://pycanon.readthedocs.io/en/stable/)
- [ARX anonymization tool](https://arx.deidentifier.org/)
- [mybinder.org](https://mybinder.org/)